# 🌾 Crop Yield Prediction — Improved Machine Learning Pipeline

**Dataset:** Crop Yield Prediction Dataset (Kaggle: `patelris/crop-yield-prediction-dataset`)
**Goal:** Predict crop yield (`hg/ha_yield`, where 1 t/ha = 10,000 hg/ha)

### What changed compared to the first version

| # | Issue in the first version | Fix in this version |
|---|---|---|
| 1 | Train/test split ran on data that **still contained the 2,310 duplicate rows** (`22,593 + 5,649 = 28,242`), so identical rows were in both sets → inflated R² | Duplicates removed first, with an `assert` guard before the split |
| 2 | Random split: same country/year/weather appears in train **and** test for other crops | **Time-based split** (train ≤ 2008, test 2009–2013) + `GroupKFold` by year inside tuning |
| 3 | Linear models could not learn the *(country × crop)* interaction | New feature `Area_Item`, encoded with a cross-fitted **target encoder** (one column, no leakage, fast) |
| 4 | Target is heavily right-skewed (5k → 500k hg/ha) | `log1p` on the target via `TransformedTargetRegressor`, `log1p` on pesticides |
| 5 | Outliers were only plotted, no decision | Per-crop outlier analysis + explicit decision |
| 6 | Boosting models used near-default settings; grid search returned the defaults | `RandomizedSearchCV` on Random Forest **and** XGBoost with wider ranges |
| 7 | Header said `yield_tpha` but the column is `hg/ha_yield` | Units fixed, MAE also reported in t/ha |
| 8 | Best model was hard-coded | Best model is picked automatically from the results table |

## 0️⃣ Setup

In [ ]:
import os
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.base import clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler, TargetEncoder
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_theme(style='whitegrid')

RANDOM_STATE = 42
FAST_MODE = False      # True = tiny/quick run just to test that the notebook executes

import sklearn

print("scikit-learn", sklearn.__version__, "(TargetEncoder needs >= 1.3)")
print("Libraries imported successfully!")

## 1️⃣ Data Loading

In [ ]:
try:
    import kagglehub
    path = kagglehub.dataset_download('patelris/crop-yield-prediction-dataset')
    print('Path to dataset files:', path)
    csv_path = os.path.join(path, 'yield_df.csv')
except Exception as e:
    # fallback: put yield_df.csv next to the notebook
    print('kagglehub not available, using local file. Reason:', e)
    csv_path = 'yield_df.csv'

df = pd.read_csv(csv_path)
if 'Unnamed: 0' in df.columns:
    df.drop(columns=['Unnamed: 0'], inplace=True)

print('Shape:', df.shape)
df.head()

## 2️⃣ Data Cleaning

**Important:** duplicates are removed *before* any split. If the split is done first, identical rows end up in both train and test and the test score becomes over-optimistic.

In [ ]:
df.columns = df.columns.str.strip()

print("--- Missing Values per Column ---")
print(df.isnull().sum())
print(f"\n--- Total Duplicate Rows: {df.duplicated().sum()} ---")
print("\n--- Data Types ---")
print(df.dtypes)

In [ ]:
df_raw = df.copy()          # kept ONLY to demonstrate the leakage effect in section 8
n_before = len(df)

df = df.drop_duplicates().reset_index(drop=True)
print(f"Rows before: {n_before} | after removing duplicates: {len(df)} | removed: {n_before - len(df)}")
assert df.duplicated().sum() == 0

TARGET = 'hg/ha_yield'
numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print("\nNumerical features:", numeric_cols)
print("Categorical features:", categorical_cols)

# sanity checks for invalid values
print("\nNegative values per numeric column:")
print((df[numeric_cols] < 0).sum())
print(f"\nYears: {df['Year'].min()} - {df['Year'].max()} | Countries: {df['Area'].nunique()} | Crops: {df['Item'].nunique()}")
df.head()

## 3️⃣ Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df[TARGET], kde=True, color='teal', bins=40, ax=axes[0])
axes[0].set_title('Yield (hg/ha) — heavily right-skewed')
sns.histplot(np.log1p(df[TARGET]), kde=True, color='darkorange', bins=40, ax=axes[1])
axes[1].set_title('log1p(Yield) — much closer to symmetric')
plt.tight_layout(); plt.show()

plt.figure(figsize=(8, 6))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix of Numerical Features')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

df.groupby('Item')[TARGET].mean().sort_values(ascending=False).plot(kind='bar', ax=axes[0], color='coral')
axes[0].set_title('Average Yield by Crop Type'); axes[0].set_ylabel('Mean Yield (hg/ha)')
axes[0].tick_params(axis='x', rotation=45)

df.groupby('Area')[TARGET].mean().sort_values(ascending=False).head(10).plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('Top 10 Countries by Mean Yield'); axes[1].set_ylabel('Mean Yield (hg/ha)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

yearly = df.groupby('Year')[TARGET].mean().reset_index()
sns.lineplot(data=yearly, x='Year', y=TARGET, marker='o', ax=axes[0], color='navy')
axes[0].set_title('Global Mean Yield Over Time')

sns.scatterplot(data=df, x='pesticides_tonnes', y=TARGET, alpha=0.3, ax=axes[1], color='crimson')
axes[1].set_title('Pesticides (Tonnes) vs Yield')

sns.scatterplot(data=df, x='average_rain_fall_mm_per_year', y=TARGET, alpha=0.3, ax=axes[2], color='seagreen')
axes[2].set_title('Rainfall (mm/year) vs Yield')

plt.tight_layout(); plt.show()

### 3.1 Outlier Analysis

The first version only drew boxplots. Here we **look deeper and take a decision**.

In [ ]:
outlier_cols = ['average_rain_fall_mm_per_year', 'pesticides_tonnes', 'avg_temp', TARGET]

plt.figure(figsize=(14, 4))
for i, col in enumerate(outlier_cols, 1):
    plt.subplot(1, 4, i)
    sns.boxplot(y=df[col], color='skyblue')
    plt.title(col, fontsize=10)
plt.tight_layout(); plt.show()

**Problem with the plot above:** the yield boxplot mixes all crops together. Potatoes, cassava, sweet potatoes and yams naturally have much higher yields than wheat or soybeans, so they look like outliers only because they are compared with *other* crops. Let's check **inside each crop**.

In [ ]:
plt.figure(figsize=(14, 5))
sns.boxplot(data=df, x='Item', y=TARGET)
plt.xticks(rotation=45); plt.title('Yield distribution per crop')
plt.tight_layout(); plt.show()

In [ ]:
def iqr_bounds(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

def count_outliers(s):
    lo, hi = iqr_bounds(s)
    return int(((s < lo) | (s > hi)).sum())

rows = []
for col in outlier_cols:
    n = count_outliers(df[col])
    rows.append({'Feature': col, 'Scope': 'all rows', 'Outliers': n, '% of rows': round(100 * n / len(df), 2)})

n_in_crop = int(df.groupby('Item')[TARGET].apply(count_outliers).sum())
rows.append({'Feature': TARGET, 'Scope': 'inside each crop', 'Outliers': n_in_crop,
             '% of rows': round(100 * n_in_crop / len(df), 2)})
pd.DataFrame(rows)

In [ ]:
# Pesticides: is it a per-row value or a country-year value?
print("Max distinct pesticide values per (Area, Year):",
      df.groupby(['Area', 'Year'])['pesticides_tonnes'].nunique().max())
print("\nTop 5 countries by mean pesticides (tonnes):")
print(df.groupby('Area')['pesticides_tonnes'].mean().sort_values(ascending=False).head(5).round(0))

# Temperature outliers: which countries?
lo, hi = iqr_bounds(df['avg_temp'])
mask = (df['avg_temp'] < lo) | (df['avg_temp'] > hi)
print("\nTemperature outliers by country:")
print(df.loc[mask].groupby('Area')['avg_temp'].agg(['count', 'min', 'max']))

### Decision on outliers

- **Rainfall:** no outliers.
- **Temperature:** only a handful of very cold country-years → real values, kept.
- **Pesticides:** right-skewed because a few large agricultural countries use far more pesticides. The value is per *(country, year)* and is repeated for every crop, which inflates the visible outlier count. Real values → **kept**, handled with `log1p`.
- **Yield:** most "outliers" are just high-yield crops. Removing them would delete whole crops from the data → **kept**, handled with `log1p` on the target.
- Tree models (Random Forest, XGBoost) are not sensitive to outliers; the log transform helps the linear models and stabilizes the errors.

## 4️⃣ Feature Engineering & Time-Based Split

- `Area_Item` = country + crop. The yield depends mostly on this *combination*, and linear models cannot build it from two separate one-hot features. It has ~1,000 distinct values, so instead of one-hot we use `TargetEncoder` (replaces each combination by its mean log-yield, computed with internal cross-fitting so the training rows never see their own target).
- **Time-based split:** train on 1990–2008, test on 2009–2013. This is closer to the real use case (predicting years the model has not seen) and avoids the leakage of a random split, where the same country/year/weather appears in both sets for other crops.

In [ ]:
assert df.duplicated().sum() == 0, "Run the cleaning section first (duplicates must be removed BEFORE the split)."

df['Area_Item'] = df['Area'] + '_' + df['Item']

FEATURES = ['Area', 'Item', 'Area_Item', 'Year',
            'average_rain_fall_mm_per_year', 'pesticides_tonnes', 'avg_temp']

TRAIN_END_YEAR = 2008
train_df = df[df['Year'] <= TRAIN_END_YEAR]
test_df  = df[df['Year'] >  TRAIN_END_YEAR]

X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_test,  y_test  = test_df[FEATURES],  test_df[TARGET]

print(f"Train: {len(train_df)} rows ({train_df['Year'].min()}-{train_df['Year'].max()})")
print(f"Test : {len(test_df)} rows ({test_df['Year'].min()}-{test_df['Year'].max()})  -> {len(test_df)/len(df):.1%} of data")

unseen = ~test_df['Area_Item'].isin(train_df['Area_Item'])
print(f"Test rows whose (country, crop) combination never appears in train: {unseen.sum()}")

In [ ]:
onehot_features = ['Area', 'Item']
te_features  = ['Area_Item']
num_features = ['Year', 'average_rain_fall_mm_per_year', 'avg_temp']
log_features = ['pesticides_tonnes']

preprocessor = ColumnTransformer(transformers=[
    ('num',    StandardScaler(), num_features),
    ('pest',   Pipeline([('log', FunctionTransformer(np.log1p, feature_names_out='one-to-one')),
                         ('scale', StandardScaler())]), log_features),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False), onehot_features),
    ('te',     TargetEncoder(target_type='continuous', random_state=RANDOM_STATE), te_features),
])

def make_model(estimator):
    """preprocessing -> estimator, trained on log1p(target), predictions returned in original units."""
    return TransformedTargetRegressor(
        regressor=Pipeline([('prep', clone(preprocessor)), ('model', estimator)]),
        func=np.log1p, inverse_func=np.expm1
    )

def evaluate(name, model, X, y):
    pred = np.clip(model.predict(X), 0, None)
    mae  = mean_absolute_error(y, pred)
    return {
        'Model': name,
        'R2': round(r2_score(y, pred), 4),
        'R2 (log scale)': round(r2_score(np.log1p(y), np.log1p(pred)), 4),
        'MAE (hg/ha)': round(mae, 0),
        'MAE (t/ha)': round(mae / 10000, 2),
        'RMSE (hg/ha)': round(float(np.sqrt(mean_squared_error(y, pred))), 0),
    }

print("Helpers ready.")

## 5️⃣ Baseline Models
All models are trained on the **log-transformed target** and evaluated on the **untouched test years**.

In [ ]:
n_est = 50 if FAST_MODE else 300

baselines = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression':  Ridge(alpha=1.0),
    'Random Forest':     RandomForestRegressor(n_estimators=(30 if FAST_MODE else 200), random_state=RANDOM_STATE, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=n_est, learning_rate=0.1, max_depth=5, random_state=RANDOM_STATE),
    'XGBoost':           XGBRegressor(n_estimators=n_est, learning_rate=0.05, max_depth=8, random_state=RANDOM_STATE, n_jobs=-1),
}

fitted, results = {}, []
for name, est in baselines.items():
    model = make_model(est).fit(X_train, y_train)
    fitted[name] = model
    results.append(evaluate(name, model, X_test, y_test))
    print(f"done: {name}")

baseline_df = pd.DataFrame(results).sort_values('R2', ascending=False).reset_index(drop=True)
baseline_df

## 6️⃣ Hyperparameter Tuning

`RandomizedSearchCV` on the two strongest tree models. The cross-validation uses **`GroupKFold` by year**, so the same year never appears in both the training and validation part of a fold.

In [ ]:
cv = GroupKFold(n_splits=2 if FAST_MODE else 3)
groups = X_train['Year']
n_iter = 2 if FAST_MODE else 8

# ---- Random Forest
rf_search = RandomizedSearchCV(
    make_model(RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)),
    param_distributions={
        'regressor__model__n_estimators': [30] if FAST_MODE else [150, 300],
        'regressor__model__max_depth': [None, 30],
        'regressor__model__min_samples_leaf': [1, 2, 4],
        'regressor__model__max_features': [0.3, 0.5],
    },
    n_iter=n_iter, cv=cv, scoring='r2', random_state=RANDOM_STATE, n_jobs=1, verbose=1
)
rf_search.fit(X_train, y_train, groups=groups)
print("Best RF params:", rf_search.best_params_)
print("Best RF CV R2 :", round(rf_search.best_score_, 4))

In [ ]:
# ---- XGBoost
xgb_search = RandomizedSearchCV(
    make_model(XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1)),
    param_distributions={
        'regressor__model__n_estimators': [50] if FAST_MODE else [400, 800, 1200],
        'regressor__model__learning_rate': [0.03, 0.05, 0.1],
        'regressor__model__max_depth': [6, 8, 10],
        'regressor__model__subsample': [0.7, 0.85, 1.0],
        'regressor__model__colsample_bytree': [0.5, 0.7, 1.0],
        'regressor__model__min_child_weight': [1, 3, 5],
    },
    n_iter=(2 if FAST_MODE else 15), cv=cv, scoring='r2', random_state=RANDOM_STATE, n_jobs=1, verbose=1
)
xgb_search.fit(X_train, y_train, groups=groups)
print("Best XGB params:", xgb_search.best_params_)
print("Best XGB CV R2 :", round(xgb_search.best_score_, 4))

## 7️⃣ Final Comparison on the Test Years

In [ ]:
fitted['Random Forest (tuned)'] = rf_search.best_estimator_
fitted['XGBoost (tuned)']       = xgb_search.best_estimator_

results.append(evaluate('Random Forest (tuned)', fitted['Random Forest (tuned)'], X_test, y_test))
results.append(evaluate('XGBoost (tuned)',       fitted['XGBoost (tuned)'],       X_test, y_test))

results_df = pd.DataFrame(results).sort_values('R2', ascending=False).reset_index(drop=True)
display(results_df)

best_name  = results_df.loc[0, 'Model']
best_model = fitted[best_name]
print(f"\n🏆 Best model on unseen years: {best_name}")

plt.figure(figsize=(8, 4))
sns.barplot(data=results_df, y='Model', x='R2', palette='viridis')
plt.title('R² on test years (2009-2013)'); plt.xlim(0, 1)
plt.tight_layout(); plt.show()

## 8️⃣ Why the split matters (leakage demonstration)

Same Random Forest, three different setups. This shows how much a careless split inflates the score.

In [ ]:
from sklearn.model_selection import train_test_split


def rf_score(frame, split):
    frame = frame.copy()
    frame['Area_Item'] = frame['Area'] + '_' + frame['Item']
    if split == 'random':
        tr, te = train_test_split(frame, test_size=0.2, random_state=RANDOM_STATE)
    else:
        tr, te = frame[frame['Year'] <= TRAIN_END_YEAR], frame[frame['Year'] > TRAIN_END_YEAR]
    m = make_model(RandomForestRegressor(n_estimators=(30 if FAST_MODE else 100), random_state=RANDOM_STATE, n_jobs=-1))
    m.fit(tr[FEATURES], tr[TARGET])
    return r2_score(te[TARGET], m.predict(te[FEATURES]))

leak_df = pd.DataFrame({
    'Setup': ['Random split, duplicates NOT removed (first version)',
              'Random split, duplicates removed',
              'Time-based split, duplicates removed (this notebook)'],
    'RF R2': [rf_score(df_raw, 'random'), rf_score(df, 'random'), rf_score(df, 'time')],
}).round(4)
leak_df

## 9️⃣ Feature Importance

In [ ]:
# importance only exists for tree models -> use the best-scoring tree-based model
tree_models = [m for m in results_df['Model']
               if hasattr(fitted[m].regressor_.named_steps['model'], 'feature_importances_')]
imp_name  = tree_models[0]
imp_model = fitted[imp_name]
print("Feature importance of:", imp_name)

inner_prep  = imp_model.regressor_.named_steps['prep']
inner_model = imp_model.regressor_.named_steps['model']

names = inner_prep.get_feature_names_out()
imp = pd.DataFrame({'Feature': names, 'Importance': inner_model.feature_importances_})

def family(f):
    if f.startswith('te__'):            return 'Area_Item (country x crop)'
    if f.startswith('onehot__Area_'):   return 'Area (country)'
    if f.startswith('onehot__Item_'):   return 'Item (crop)'
    return f.split('__', 1)[1]

imp['Group'] = imp['Feature'].map(family)
grouped = imp.groupby('Group')['Importance'].sum().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
grouped.plot(kind='barh', ax=axes[0], color='teal'); axes[0].invert_yaxis()
axes[0].set_title('Importance by feature group')
top = imp.sort_values('Importance', ascending=False).head(12)
sns.barplot(data=top, x='Importance', y='Feature', palette='mako', ax=axes[1])
axes[1].set_title('Top 12 individual features')
plt.tight_layout(); plt.show()

## 🔟 Residual Diagnostics & Errors per Crop

In [ ]:
pred_best = np.clip(best_model.predict(X_test), 0, None)
log_resid = np.log1p(y_test) - np.log1p(pred_best)

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
axes[0].scatter(y_test, pred_best, alpha=0.3, color='forestgreen')
lim = [min(y_test.min(), pred_best.min()) + 1, max(y_test.max(), pred_best.max())]
axes[0].plot(lim, lim, '--r'); axes[0].set_xscale('log'); axes[0].set_yscale('log')
axes[0].set_title('Actual vs Predicted (log-log)'); axes[0].set_xlabel('Actual'); axes[0].set_ylabel('Predicted')

axes[1].scatter(np.log1p(pred_best), log_resid, alpha=0.3, color='teal')
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_title('Residuals (log scale) vs Predicted'); axes[1].set_xlabel('log1p(Predicted)')

sns.histplot(log_resid, kde=True, bins=40, ax=axes[2], color='teal')
axes[2].set_title('Residual distribution (log scale)')
plt.tight_layout(); plt.show()

In [ ]:
err = test_df[['Item', TARGET]].copy()
err['pred'] = pred_best
err['abs_err_tpha'] = (err[TARGET] - err['pred']).abs() / 10000
err['rel_err_%'] = 100 * (err[TARGET] - err['pred']).abs() / err[TARGET].clip(lower=1)

per_crop = err.groupby('Item').agg(
    mean_yield_tpha=(TARGET, lambda s: s.mean() / 10000),
    MAE_tpha=('abs_err_tpha', 'mean'),
    median_rel_err_pct=('rel_err_%', 'median'),
).round(2).sort_values('mean_yield_tpha', ascending=False)
per_crop

## 1️⃣1️⃣ Save the Final Model

The best configuration is **re-fitted on all years (1990-2013)** before saving, because more (and more recent) data helps the deployed model. The scores above stay valid: they came from the model that never saw 2009-2013.

In [ ]:
final_model = clone(best_model).fit(df[FEATURES], df[TARGET])

artifact = {
    'model': final_model,
    'features': FEATURES,
    'known_areas': sorted(df['Area'].unique()),
    'known_items': sorted(df['Item'].unique()),
    'best_model_name': best_name,
    'test_metrics': results_df[results_df['Model'] == best_name].iloc[0].to_dict(),
}
joblib.dump(artifact, 'crop_yield_pipeline.joblib')
print("Saved: crop_yield_pipeline.joblib")

## 1️⃣2️⃣ Prediction Function

In [ ]:
loaded = joblib.load('crop_yield_pipeline.joblib')

def predict_crop_yield(area, item, year, rain_fall, pesticides, temp, artifact=loaded, model=None):
    """Predict yield for raw inputs. Returns hg/ha and t/ha."""
    if area not in artifact['known_areas']:
        raise ValueError(f"Unknown country '{area}'. Known countries: {len(artifact['known_areas'])} (see artifact['known_areas']).")
    if item not in artifact['known_items']:
        raise ValueError(f"Unknown crop '{item}'. Choose from: {artifact['known_items']}")

    X_new = pd.DataFrame([{
        'Area': area, 'Item': item, 'Area_Item': f'{area}_{item}', 'Year': year,
        'average_rain_fall_mm_per_year': rain_fall,
        'pesticides_tonnes': pesticides, 'avg_temp': temp,
    }])[artifact['features']]

    pred = float(np.clip((model or artifact['model']).predict(X_new)[0], 0, None))
    return {'hg_per_ha': round(pred, 1), 't_per_ha': round(pred / 10000, 3)}

# Honest sanity check: rows from the TEST years, predicted by the model that never saw them
sample = test_df.sample(6, random_state=RANDOM_STATE)
rows = []
for _, r in sample.iterrows():
    p = predict_crop_yield(r['Area'], r['Item'], r['Year'], r['average_rain_fall_mm_per_year'],
                           r['pesticides_tonnes'], r['avg_temp'], model=best_model)
    rows.append({'Area': r['Area'], 'Item': r['Item'], 'Year': r['Year'],
                 'Actual (t/ha)': round(r[TARGET] / 10000, 3), 'Predicted (t/ha)': p['t_per_ha']})
pd.DataFrame(rows)

In [ ]:
# Example call with the deployed (all-years) model
predict_crop_yield(area='Albania', item='Maize', year=2013, rain_fall=1485.0, pesticides=121.0, temp=16.37)

## ✅ Summary & Limitations

- Duplicates are removed **before** splitting, and the model is evaluated on **unseen years** — the reported scores are realistic, not inflated.
- The main drivers of yield are the **country × crop** combination and the year (technology trend); weather and pesticide values add smaller refinements.
- Rainfall, temperature and pesticides are **country-level yearly values** (identical for all crops in a country-year), so the model predicts national average yields, not field-level yields.
- The model can only predict countries and crops that exist in the data; the prediction function raises a clear error for anything else.
- Trees cannot extrapolate: for years far beyond 2013 the trend will be flat.